# MLVerse-Math — Probability Theory
## `02_Events` — Interactive Visualization Notebook

> **Learning path:** Beginner → Intermediate → Advanced  
> **Runtime:** Python 3.12+  
> **Audience:** Students, AI engineers, researchers, and educators

An **event** is a subset of a sample space:

$$
E \subseteq S
$$

This notebook visualizes Events through:

- interactive Plotly charts,
- Matplotlib animations,
- `ipywidgets` controls,
- Venn diagrams,
- NetworkX trees,
- Monte Carlo simulations,
- AI, NLP, Computer Vision, and LLM examples.

Every major visualization begins with intuition, motivation, mathematics, AI context, and observations.

## 0. Environment, Imports, and Compatibility

The requested backend is:

```python
%matplotlib widget
```

The setup cell:

1. checks all required libraries,
2. automatically installs only missing packages,
3. tries the interactive widget backend,
4. falls back to inline Matplotlib output if `ipympl` is unavailable.

> This design improves portability across Jupyter Notebook, JupyterLab, VS Code, Google Colab, and Kaggle while preserving richer interactivity where supported.

In [ ]:
from __future__ import annotations

import importlib
import itertools
import math
import random
import subprocess
import sys
from dataclasses import dataclass
from typing import Callable, Iterable, Sequence

# -------------------------------------------------------------------
# Dependency bootstrap
# -------------------------------------------------------------------
# The notebook uses a broad visualization stack. Most Jupyter/Colab/
# Kaggle environments already provide these packages. Missing packages
# are installed automatically so the notebook can run without editing.
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "plotly": "plotly",
    "matplotlib_venn": "matplotlib-venn",
    "networkx": "networkx",
    "ipywidgets": "ipywidgets",
    "scipy": "scipy",
    "sympy": "sympy",
}

missing_packages = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing_packages.append(package_name)

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing_packages]
    )

# Numerical and symbolic stack
import numpy as np
import pandas as pd
import scipy.stats as stats
import sympy as sp

# Visualization stack
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Circle, Rectangle
from matplotlib_venn import venn2

import plotly.express as px
import plotly.graph_objects as go

# Graphs and interactivity
import networkx as nx
import ipywidgets as widgets
from IPython.display import HTML, Markdown, clear_output, display

RNG = np.random.default_rng(42)
random.seed(42)

plt.rcParams.update({
    "figure.figsize": (14, 10),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "font.size": 11,
})

# Try the requested interactive widget backend.
# If ipympl is unavailable, fall back cleanly to inline rendering.
try:
    get_ipython().run_line_magic("matplotlib", "widget")
    MATPLOTLIB_BACKEND = "widget"
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")
    MATPLOTLIB_BACKEND = "inline"

print(f"Matplotlib backend: {MATPLOTLIB_BACKEND}")
print("Dependencies ready.")

In [ ]:
@dataclass(frozen=True)
class FiniteEvent:
    """Named event inside a finite sample space."""

    name: str
    outcomes: frozenset

    def probability(self, sample_space: Sequence) -> float:
        if len(sample_space) == 0:
            raise ValueError("Sample space must not be empty.")
        return len(self.outcomes) / len(sample_space)


def event_probability(event: Iterable, sample_space: Sequence) -> float:
    """Compute |E| / |S| for a finite equally likely sample space."""
    sample = set(sample_space)
    selected = set(event)
    if not selected.issubset(sample):
        raise ValueError("Event must be a subset of the sample space.")
    return len(selected) / len(sample)


def powerset_event_space(sample_space: Sequence) -> list[frozenset]:
    """Return 2^S for a small finite sample space."""
    items = list(sample_space)
    return [
        frozenset(combo)
        for size in range(len(items) + 1)
        for combo in itertools.combinations(items, size)
    ]


def operation_result(
    event_a: set,
    event_b: set,
    sample_space: set,
    operation: str,
) -> set:
    """Apply standard event-algebra operations."""
    mapping = {
        "Union A ∪ B": event_a | event_b,
        "Intersection A ∩ B": event_a & event_b,
        "Difference A − B": event_a - event_b,
        "Difference B − A": event_b - event_a,
        "Complement Aᶜ": sample_space - event_a,
        "Complement Bᶜ": sample_space - event_b,
        "Symmetric Difference": event_a ^ event_b,
    }
    if operation not in mapping:
        raise ValueError(f"Unknown operation: {operation}")
    return mapping[operation]


def style_plotly(fig: go.Figure, title: str, height: int = 600) -> go.Figure:
    """Apply a consistent publication-quality Plotly theme."""
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=height,
        margin=dict(l=40, r=40, t=85, b=45),
        legend_title_text="",
        hoverlabel=dict(font_size=12),
    )
    return fig

---
        # 1. Introduction to Events

        ### Learning Objective
        Understand the hierarchy from random experiment to sample space, outcome, and event.

        ### Mathematical Intuition

A random experiment produces an outcome $\omega$ from a sample space $S$:

$$
\omega \in S
$$

An event is a collection of outcomes:

$$
E \subseteq S
$$


        ### Real-world Motivation

For weather:

$$
S=\{\text{sunny},\text{cloudy},\text{rainy},\text{stormy}\}
$$

The event “wet weather” is:

$$
E=\{\text{rainy},\text{stormy}\}
$$


        ### AI Connection

A classifier may have:

$$
S=\{\text{cat},\text{dog},\text{car},\text{person}\}
$$

An event can group several output classes.


        ### Key Observations
        - Outcomes are individual elements.
- Events are subsets.
- One sample space supports many events.

In [ ]:
stages = [
    ("Random Experiment", "Perform an uncertain process"),
    ("Sample Space S", "Collect every possible outcome"),
    ("Outcome ω", "Observe one element of S"),
    ("Event E ⊆ S", "Select outcomes of interest"),
]

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

artists = []
for index, (title, subtitle) in enumerate(stages):
    y_value = 8.5 - index * 2.1
    box = Rectangle((1.5, y_value - 0.65), 7.0, 1.3, alpha=0.08)
    ax.add_patch(box)
    label = ax.text(
        5,
        y_value,
        f"{title}\n{subtitle}",
        ha="center",
        va="center",
        fontsize=16,
        alpha=0.15,
    )
    artists.append((box, label))

def update_intro(frame: int):
    for index, (box, label) in enumerate(artists):
        visible = index <= frame
        box.set_alpha(0.20 if visible else 0.05)
        label.set_alpha(1.0 if visible else 0.15)
    ax.set_title("From Random Experiments to Events", fontsize=20, pad=20)
    return [item for pair in artists for item in pair]

intro_animation = animation.FuncAnimation(
    fig,
    update_intro,
    frames=len(stages),
    interval=900,
    repeat=True,
)

plt.close(fig)
HTML(intro_animation.to_jshtml())

---
        # 2. Coin Toss Event Visualization

        ### Learning Objective
        Explore event subsets for one, two, and three tosses.

        ### Mathematical Intuition

For $n$ tosses:

$$
S_n=\{H,T\}^n,
\qquad
|S_n|=2^n
$$


        ### Real-world Motivation
        Coin models abstract success/failure, click/no-click, and defect/no-defect processes.

        ### AI Connection

Binary classification uses the same two-outcome idea:

$$
S=\{0,1\}
$$


        ### Key Observations
        - Larger $n$ expands the sample space exponentially.
- Event predicates select subsets dynamically.

In [ ]:
def coin_outcomes(n_tosses: int) -> list[str]:
    """Generate all H/T sequences of length n_tosses."""
    return ["".join(p) for p in itertools.product("HT", repeat=n_tosses)]


def select_coin_event(
    outcomes: Sequence[str],
    event_name: str,
) -> set[str]:
    """Select coin outcomes satisfying a named event."""
    predicates: dict[str, Callable[[str], bool]] = {
        "Exactly one Head": lambda x: x.count("H") == 1,
        "At least one Head": lambda x: x.count("H") >= 1,
        "All Heads": lambda x: x.count("H") == len(x),
        "No Heads": lambda x: x.count("H") == 0,
    }
    return {x for x in outcomes if predicates[event_name](x)}


def plot_coin_event(
    n_tosses: int = 2,
    event_name: str = "At least one Head",
) -> go.Figure:
    outcomes = coin_outcomes(n_tosses)
    selected = select_coin_event(outcomes, event_name)

    df = pd.DataFrame({
        "Outcome": outcomes,
        "Selected": [item in selected for item in outcomes],
    })
    df["Status"] = np.where(df["Selected"], "In Event", "Outside Event")
    df["Index"] = np.arange(len(df))

    fig = px.scatter(
        df,
        x="Index",
        y=np.ones(len(df)),
        text="Outcome",
        color="Status",
        size=np.where(df["Selected"], 22, 10),
        hover_data=["Outcome", "Selected"],
        color_discrete_sequence=px.colors.sequential.Viridis[2:8:4],
    )
    fig.update_traces(textposition="top center")
    fig.update_yaxes(visible=False)
    fig.update_xaxes(title="Outcome index")
    probability = len(selected) / len(outcomes)
    return style_plotly(
        fig,
        f"{n_tosses} Toss(es): {event_name} — P(E)={probability:.3f}",
    )


coin_count = widgets.IntSlider(
    value=2,
    min=1,
    max=3,
    step=1,
    description="Tosses:",
)
coin_event = widgets.Dropdown(
    options=[
        "Exactly one Head",
        "At least one Head",
        "All Heads",
        "No Heads",
    ],
    value="At least one Head",
    description="Event:",
)

coin_view = widgets.interactive_output(
    lambda tosses, event: plot_coin_event(tosses, event).show(),
    {"tosses": coin_count, "event": coin_event},
)

display(widgets.HBox([coin_count, coin_event]), coin_view)

---
        # 3. Dice Event Explorer

        ### Learning Objective
        Interactively select common events from a fair six-sided die.

        ### Mathematical Intuition

$$
S=\{1,2,3,4,5,6\}
$$

Examples:

$$
E_{\text{even}}=\{2,4,6\},
\qquad
E_{\text{prime}}=\{2,3,5\}
$$


        ### Real-world Motivation
        Finite event filters model eligibility rules, thresholds, and categories.

        ### AI Connection

AI systems often define events by predicates:

$$
E=\{x:f(x)>\tau\}
$$


        ### Key Observations
        - Events are logical filters.
- Different events may overlap.
- Membership can be highlighted dynamically.

In [ ]:
DIE_SPACE = [1, 2, 3, 4, 5, 6]

DIE_EVENTS = {
    "Even Numbers": {2, 4, 6},
    "Odd Numbers": {1, 3, 5},
    "Prime Numbers": {2, 3, 5},
    "Multiples of 3": {3, 6},
    "Numbers Greater than 4": {5, 6},
}


def dice_event_figure(event_name: str) -> go.Figure:
    event = DIE_EVENTS[event_name]
    membership = [1 if value in event else 0 for value in DIE_SPACE]

    fig = go.Figure(go.Bar(
        x=[str(value) for value in DIE_SPACE],
        y=[1] * len(DIE_SPACE),
        marker=dict(
            color=membership,
            colorscale="Plasma",
            showscale=True,
            colorbar=dict(title="Membership"),
        ),
        text=[
            f"{value}<br>{'IN E' if value in event else 'OUT'}"
            for value in DIE_SPACE
        ],
        textposition="inside",
        hovertemplate="Outcome=%{x}<extra></extra>",
    ))
    fig.update_yaxes(visible=False)
    fig.update_xaxes(title="Die outcome")
    return style_plotly(
        fig,
        f"Dice Event Explorer: {event_name} — E={sorted(event)}",
    )


widgets.interact(
    lambda event_name: dice_event_figure(event_name).show(),
    event_name=widgets.Dropdown(
        options=list(DIE_EVENTS),
        value="Even Numbers",
        description="Event:",
    ),
)

---
        # 4. Playing Card Event Visualization

        ### Learning Objective
        See named card events as subsets of a 52-card sample space.

        ### Mathematical Intuition

Let $S$ be all 52 standard cards.

$$
E_{\text{hearts}}=\{\text{all 13 hearts}\}
$$

$$
E_{\text{kings}}=\{K\spadesuit,K\heartsuit,K\diamondsuit,K\clubsuit\}
$$


        ### Real-world Motivation
        Card decks illustrate sample spaces with multiple categorical attributes.

        ### AI Connection
        A data point can simultaneously belong to events defined by color, class, rank, or semantic category.

        ### Key Observations
        - A single outcome can belong to multiple events.
- Event overlap is natural in multi-attribute spaces.

In [ ]:
SUITS = ["♠", "♥", "♦", "♣"]
RANKS = ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]
RED_SUITS = {"♥", "♦"}

deck = pd.DataFrame(
    [(rank, suit) for suit in SUITS for rank in RANKS],
    columns=["Rank", "Suit"],
)
deck["Card"] = deck["Rank"] + deck["Suit"]
deck["Color"] = np.where(deck["Suit"].isin(RED_SUITS), "Red", "Black")


def card_event_mask(df: pd.DataFrame, event_name: str) -> pd.Series:
    conditions = {
        "Red Cards": df["Color"].eq("Red"),
        "Black Cards": df["Color"].eq("Black"),
        "Hearts": df["Suit"].eq("♥"),
        "Face Cards": df["Rank"].isin(["J", "Q", "K"]),
        "Kings": df["Rank"].eq("K"),
        "Aces": df["Rank"].eq("A"),
    }
    return conditions[event_name]


def card_event_figure(event_name: str) -> go.Figure:
    df = deck.copy()
    df["Selected"] = card_event_mask(df, event_name)
    df["Row"] = df["Suit"].map({suit: i for i, suit in enumerate(SUITS)})
    df["Col"] = df["Rank"].map({rank: i for i, rank in enumerate(RANKS)})

    fig = px.scatter(
        df,
        x="Col",
        y="Row",
        text="Card",
        color="Selected",
        size=np.where(df["Selected"], 22, 12),
        hover_data=["Card", "Color", "Selected"],
        color_discrete_sequence=px.colors.sequential.Cividis[1:8:5],
    )
    fig.update_traces(textposition="middle center")
    fig.update_xaxes(
        tickmode="array",
        tickvals=list(range(len(RANKS))),
        ticktext=RANKS,
        title="Rank",
    )
    fig.update_yaxes(
        tickmode="array",
        tickvals=list(range(len(SUITS))),
        ticktext=SUITS,
        title="Suit",
    )

    count = int(df["Selected"].sum())
    return style_plotly(
        fig,
        f"Card Event: {event_name} — |E|={count}, P(E)={count/52:.3f}",
    )


widgets.interact(
    lambda event_name: card_event_figure(event_name).show(),
    event_name=widgets.Dropdown(
        options=["Red Cards", "Black Cards", "Hearts", "Face Cards", "Kings", "Aces"],
        description="Event:",
    ),
)

---
        # 5. Venn Diagram Explorer

        ### Learning Objective
        Interpret union, intersection, difference, and overlap geometrically.

        ### Mathematical Intuition

For $A,B\subseteq S$:

$$
A\cup B,\qquad A\cap B,\qquad A-B
$$


        ### Real-world Motivation
        Venn regions represent overlapping customer segments, risk groups, and medical symptoms.

        ### AI Connection

Model events may overlap:

$$
A=\{\text{high confidence}\},
\qquad
B=\{\text{positive prediction}\}
$$


        ### Key Observations
        - Overlap means simultaneous membership.
- Complements must be interpreted relative to $S$.

In [ ]:
VENN_SAMPLE_SPACE = set(range(1, 11))
VENN_A = {1, 2, 3, 4, 5, 6}
VENN_B = {4, 5, 6, 7, 8}


def plot_venn_operation(operation: str) -> None:
    result = operation_result(
        VENN_A,
        VENN_B,
        VENN_SAMPLE_SPACE,
        operation,
    )

    fig, ax = plt.subplots(figsize=(14, 10))
    diagram = venn2(
        subsets=(
            len(VENN_A - VENN_B),
            len(VENN_B - VENN_A),
            len(VENN_A & VENN_B),
        ),
        set_labels=("Event A", "Event B"),
        ax=ax,
    )

    for region_id in ("10", "01", "11"):
        patch = diagram.get_patch_by_id(region_id)
        if patch is not None:
            patch.set_alpha(0.15)

    selected_regions = {
        "Union A ∪ B": {"10", "01", "11"},
        "Intersection A ∩ B": {"11"},
        "Difference A − B": {"10"},
        "Difference B − A": {"01"},
        "Symmetric Difference": {"10", "01"},
    }.get(operation, set())

    for region_id in selected_regions:
        patch = diagram.get_patch_by_id(region_id)
        if patch is not None:
            patch.set_alpha(0.85)

    ax.set_title(f"{operation}\nResult={sorted(result)}", fontsize=18)
    plt.show()


widgets.interact(
    plot_venn_operation,
    operation=widgets.Dropdown(
        options=[
            "Union A ∪ B",
            "Intersection A ∩ B",
            "Difference A − B",
            "Difference B − A",
            "Symmetric Difference",
        ],
        description="Operation:",
    ),
)

---
        # 6. Event Algebra Animation

        ### Learning Objective
        Watch the main set operations appear as changing geometric regions.

        ### Mathematical Intuition

$$
A\cup B,\qquad
A\cap B,\qquad
A-B,\qquad
A^c
$$


        ### Real-world Motivation
        Event algebra combines and filters conditions in decision systems.

        ### AI Connection

A fraud system may use:

$$
A=\{\text{unusual amount}\},
\qquad
B=\{\text{unusual location}\}
$$

Then $A\cap B$ selects transactions satisfying both signals.


        ### Key Observations
        - Union broadens.
- Intersection narrows.
- Difference removes.
- Complement reverses membership.

In [ ]:
def create_event_algebra_animation() -> HTML:
    operations = [
        ("Union A ∪ B", "union"),
        ("Intersection A ∩ B", "intersection"),
        ("Difference A − B", "difference"),
        ("Complement Aᶜ", "complement"),
    ]

    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.axis("off")

    sample = Rectangle((0.7, 0.7), 8.6, 6.6, fill=False, linewidth=2)
    circle_a = Circle((4.0, 4.0), 2.2, alpha=0.15)
    circle_b = Circle((6.0, 4.0), 2.2, alpha=0.15)
    ax.add_patch(sample)
    ax.add_patch(circle_a)
    ax.add_patch(circle_b)
    ax.text(1.0, 6.8, "Sample Space S", fontsize=14)
    ax.text(3.0, 4.0, "A", fontsize=22)
    ax.text(6.8, 4.0, "B", fontsize=22)

    grid_x, grid_y = np.meshgrid(
        np.linspace(0.8, 9.2, 100),
        np.linspace(0.8, 7.2, 80),
    )
    x_values = grid_x.ravel()
    y_values = grid_y.ravel()

    in_a = (x_values - 4.0) ** 2 + (y_values - 4.0) ** 2 <= 2.2 ** 2
    in_b = (x_values - 6.0) ** 2 + (y_values - 4.0) ** 2 <= 2.2 ** 2

    points = ax.scatter(x_values, y_values, s=8, alpha=0.0)
    title = ax.set_title("", fontsize=20)

    def update(frame: int):
        operation_name, operation_key = operations[frame]

        if operation_key == "union":
            mask = in_a | in_b
        elif operation_key == "intersection":
            mask = in_a & in_b
        elif operation_key == "difference":
            mask = in_a & ~in_b
        else:
            mask = ~in_a

        points.set_array(np.where(mask, 1.0, 0.0))
        points.set_cmap("viridis")
        points.set_alpha(0.45)
        title.set_text(operation_name)
        return points, title

    event_animation = animation.FuncAnimation(
        fig,
        update,
        frames=len(operations),
        interval=1100,
        repeat=True,
    )
    plt.close(fig)
    return HTML(event_animation.to_jshtml())


create_event_algebra_animation()

---
        # 7. Mutually Exclusive Events

        ### Learning Objective
        Recognize event pairs that cannot occur together.

        ### Mathematical Intuition

$$
A\cap B=\emptyset
$$


        ### Real-world Motivation
        A die outcome cannot be both even and odd. One coin result cannot be both Heads and Tails.

        ### AI Connection
        Single-label output classes are often treated as disjoint prediction events.

        ### Key Observations
        > Mutually exclusive events are not the same as independent events.

In [ ]:
EXCLUSIVE_EXAMPLES = {
    "Even vs Odd": ({2, 4, 6}, {1, 3, 5}, set(range(1, 7))),
    "Heads vs Tails": ({"H"}, {"T"}, {"H", "T"}),
}


def mutually_exclusive_figure(example: str) -> go.Figure:
    event_a, event_b, sample = EXCLUSIVE_EXAMPLES[example]
    ordered = sorted(sample, key=str)

    membership = [
        "A" if item in event_a else "B" if item in event_b else "Neither"
        for item in ordered
    ]

    df = pd.DataFrame({
        "Outcome": [str(item) for item in ordered],
        "Membership": membership,
        "Value": 1,
    })

    fig = px.bar(
        df,
        x="Outcome",
        y="Value",
        color="Membership",
        color_discrete_sequence=px.colors.sequential.Inferno[2:8:2],
        hover_data=["Membership"],
    )
    fig.update_yaxes(visible=False)
    intersection = event_a & event_b
    intersection_text = sorted(intersection) if intersection else "∅"
    return style_plotly(
        fig,
        f"{example}: A ∩ B = {intersection_text}",
    )


widgets.interact(
    lambda example: mutually_exclusive_figure(example).show(),
    example=widgets.Dropdown(
        options=list(EXCLUSIVE_EXAMPLES),
        description="Example:",
    ),
)

---
        # 8. Exhaustive Events

        ### Learning Objective
        Identify event collections that completely cover the sample space.

        ### Mathematical Intuition

$$
A\cup B=S
$$


        ### Real-world Motivation
        Even and odd die outcomes cover every possible die result; red and black cover a standard deck.

        ### AI Connection
        A complete label taxonomy aims to cover all allowed outputs.

        ### Key Observations
        - Exhaustive events cover every outcome.
- Exhaustive events need not always be disjoint.

In [ ]:
EXHAUSTIVE_EXAMPLES = {
    "Even and Odd": ({2, 4, 6}, {1, 3, 5}, set(range(1, 7))),
    "Red and Black Cards": ({"Red"}, {"Black"}, {"Red", "Black"}),
}


def exhaustive_figure(example: str) -> go.Figure:
    event_a, event_b, sample = EXHAUSTIVE_EXAMPLES[example]
    union = event_a | event_b

    df = pd.DataFrame({
        "Region": ["A", "B", "A ∪ B", "S"],
        "Size": [len(event_a), len(event_b), len(union), len(sample)],
    })

    fig = px.bar(
        df,
        x="Region",
        y="Size",
        color="Size",
        color_continuous_scale="Cividis",
        text="Size",
    )
    fig.update_traces(textposition="outside")
    return style_plotly(
        fig,
        f"{example}: A ∪ B = S is {union == sample}",
    )


widgets.interact(
    lambda example: exhaustive_figure(example).show(),
    example=widgets.Dropdown(
        options=list(EXHAUSTIVE_EXAMPLES),
        description="Example:",
    ),
)

---
        # 9. Complement of an Event

        ### Learning Objective
        Automatically construct and visualize the complement of a selected event.

        ### Mathematical Intuition

For $E\subseteq S$:

$$
E^c=S-E
$$

and:

$$
E\cup E^c=S,
\qquad
E\cap E^c=\emptyset
$$


        ### Real-world Motivation
        If $E$ is “loan approved,” then $E^c$ is “loan not approved.”

        ### AI Connection

For a binary decision:

$$
E=\{\hat y=1\}
\quad\Longrightarrow\quad
E^c=\{\hat y\ne1\}
$$


        ### Key Observations
        - Complements depend on the chosen sample space.

In [ ]:
COMPLEMENT_EVENTS = {
    "Even": {2, 4, 6},
    "Prime": {2, 3, 5},
    "Greater than 3": {4, 5, 6},
    "At most 2": {1, 2},
}


def complement_figure(event_name: str) -> go.Figure:
    sample = set(DIE_SPACE)
    event = COMPLEMENT_EVENTS[event_name]
    complement = sample - event

    df = pd.DataFrame({
        "Outcome": DIE_SPACE,
        "Region": ["E" if item in event else "Eᶜ" for item in DIE_SPACE],
        "Value": 1,
    })

    fig = px.bar(
        df,
        x="Outcome",
        y="Value",
        color="Region",
        color_discrete_sequence=px.colors.sequential.Viridis[2:8:4],
    )
    fig.update_yaxes(visible=False)
    return style_plotly(
        fig,
        f"{event_name}: E={sorted(event)}, Eᶜ={sorted(complement)}",
    )


widgets.interact(
    lambda event_name: complement_figure(event_name).show(),
    event_name=widgets.Dropdown(
        options=list(COMPLEMENT_EVENTS),
        description="Event:",
    ),
)

---
        # 10. Event Probability Dashboard

        ### Learning Objective
        Compute finite classical event probabilities dynamically.

        ### Mathematical Intuition

For equally likely finite outcomes:

$$
P(E)=\frac{|E|}{|S|}
$$


        ### Real-world Motivation
        The ratio counts favorable outcomes relative to all possible outcomes.

        ### AI Connection

In learned discrete distributions:

$$
P(E)=\sum_{\omega\in E}P(\omega)
$$


        ### Key Observations
        - $|E|$ counts favorable outcomes.
- $|S|$ counts total outcomes.
- Equal likelihood is an assumption.

In [ ]:
PROBABILITY_EXPERIMENTS = {
    "Die": {
        "space": set(DIE_SPACE),
        "events": DIE_EVENTS,
    },
    "Two Coins": {
        "space": set(coin_outcomes(2)),
        "events": {
            name: select_coin_event(coin_outcomes(2), name)
            for name in [
                "Exactly one Head",
                "At least one Head",
                "All Heads",
                "No Heads",
            ]
        },
    },
    "Cards": {
        "space": set(deck["Card"]),
        "events": {
            name: set(deck.loc[card_event_mask(deck, name), "Card"])
            for name in [
                "Red Cards",
                "Black Cards",
                "Hearts",
                "Face Cards",
                "Kings",
                "Aces",
            ]
        },
    },
}

experiment_control = widgets.Dropdown(
    options=list(PROBABILITY_EXPERIMENTS),
    value="Die",
    description="Experiment:",
)
event_control = widgets.Dropdown(description="Event:")
probability_output = widgets.Output()


def refresh_probability_events(*_):
    events = PROBABILITY_EXPERIMENTS[experiment_control.value]["events"]
    event_control.options = list(events)
    event_control.value = event_control.options[0]


def render_probability_dashboard(*_):
    with probability_output:
        clear_output(wait=True)

        experiment = PROBABILITY_EXPERIMENTS[experiment_control.value]
        sample = experiment["space"]
        event = experiment["events"][event_control.value]
        probability = len(event) / len(sample)

        result_markdown = (
            "### Result\n\n"
            "$$\n"
            "P(E)=\\frac{|E|}{|S|}"
            f"=\\frac{{{len(event)}}}{{{len(sample)}}}"
            f"={probability:.4f}\n"
            "$$\n\n"
            f"- **Favorable outcomes:** `{len(event)}`\n"
            f"- **Total outcomes:** `{len(sample)}`"
        )
        display(Markdown(result_markdown))

        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=probability,
            number={"valueformat": ".4f"},
            gauge={
                "axis": {"range": [0, 1]},
                "bar": {"color": px.colors.sequential.Viridis[5]},
            },
            title={
                "text": f"{experiment_control.value} — {event_control.value}"
            },
        ))
        fig.update_layout(height=420)
        fig.show()


experiment_control.observe(refresh_probability_events, names="value")
experiment_control.observe(render_probability_dashboard, names="value")
event_control.observe(render_probability_dashboard, names="value")

refresh_probability_events()
render_probability_dashboard()

display(
    widgets.HBox([experiment_control, event_control]),
    probability_output,
)

---
        # 11. Tree Diagram Generator

        ### Learning Objective
        Represent sequential random experiments as branching paths.

        ### Mathematical Intuition

For two fair tosses:

$$
S=\{HH,HT,TH,TT\}
$$


        ### Real-world Motivation
        Trees model sequential decisions, diagnostic pathways, and multistage experiments.

        ### AI Connection
        Decision processes and reinforcement-learning trajectories can be interpreted as branching outcome histories.

        ### Key Observations
        - Nodes are partial histories.
- Leaves are complete outcomes.
- Events are sets of selected paths.

In [ ]:
def build_tree(experiment: str) -> nx.DiGraph:
    graph = nx.DiGraph()
    graph.add_node("Start")

    if experiment == "Coins":
        first_level = ["H", "T"]
        second_level = ["H", "T"]
    elif experiment == "Dice":
        first_level = ["Low", "High"]
        second_level = ["Odd", "Even"]
    else:
        first_level = ["Red", "Black"]
        second_level = ["Face", "Number"]

    for first in first_level:
        graph.add_edge("Start", first)
        for second in second_level:
            graph.add_edge(first, f"{first}→{second}")

    return graph


def layered_tree_positions(graph: nx.DiGraph) -> dict:
    """Deterministic positions without optional Graphviz dependencies."""
    positions = {"Start": (0.0, 0.0)}
    first_level = list(graph.successors("Start"))

    for index, node in enumerate(first_level):
        x_value = -1.5 if index == 0 else 1.5
        positions[node] = (x_value, -1.5)

        children = list(graph.successors(node))
        for child_index, child in enumerate(children):
            offset = -0.65 if child_index == 0 else 0.65
            positions[child] = (x_value + offset, -3.0)

    return positions


def plot_tree(
    experiment: str = "Coins",
    highlight: str = "First branch",
) -> None:
    graph = build_tree(experiment)
    positions = layered_tree_positions(graph)

    if highlight == "First branch":
        root = list(graph.successors("Start"))[0]
        selected = {root, *nx.descendants(graph, root)}
    elif highlight == "Second branch":
        root = list(graph.successors("Start"))[-1]
        selected = {root, *nx.descendants(graph, root)}
    else:
        selected = {
            node
            for node in graph.nodes
            if "→" in str(node)
        }

    node_values = [1 if node in selected else 0 for node in graph.nodes]

    fig, ax = plt.subplots(figsize=(14, 10))
    nx.draw_networkx(
        graph,
        pos=positions,
        ax=ax,
        node_color=node_values,
        cmap="viridis",
        node_size=2600,
        arrows=True,
        font_size=10,
        edge_color="gray",
    )
    ax.set_title(f"{experiment} Tree — Highlight: {highlight}")
    ax.axis("off")
    plt.show()


widgets.interact(
    plot_tree,
    experiment=widgets.Dropdown(
        options=["Coins", "Dice", "Cards"],
        description="Experiment:",
    ),
    highlight=widgets.Dropdown(
        options=["First branch", "Second branch", "All leaves"],
        description="Highlight:",
    ),
)

---
        # 12. Event Relationship Explorer

        ### Learning Objective
        Distinguish subset, equal, overlapping, and disjoint events.

        ### Mathematical Intuition

Subset:

$$
A\subseteq B
$$

Equal:

$$
A=B
$$

Overlapping:

$$
A\cap B\ne\emptyset
$$

Disjoint:

$$
A\cap B=\emptyset
$$


        ### Real-world Motivation
        These relationships describe segmentation and logical structure between criteria.

        ### AI Connection
        Confidence regions, class regions, and risk bands may be nested, overlapping, equal, or disjoint.

        ### Key Observations
        - Relationship type is determined entirely by membership structure.

In [ ]:
RELATIONSHIP_CASES = {
    "Subset": ({1, 2}, {1, 2, 3, 4}),
    "Equal Events": ({1, 2, 3}, {1, 2, 3}),
    "Overlapping Events": ({1, 2, 3}, {3, 4, 5}),
    "Disjoint Events": ({1, 2}, {4, 5}),
}


def relationship_figure(case_name: str) -> go.Figure:
    event_a, event_b = RELATIONSHIP_CASES[case_name]
    sample = sorted(event_a | event_b | {1, 2, 3, 4, 5})

    df = pd.DataFrame({
        "Outcome": sample * 2,
        "Event": ["A"] * len(sample) + ["B"] * len(sample),
        "Member": (
            [item in event_a for item in sample]
            + [item in event_b for item in sample]
        ),
    })

    fig = px.scatter(
        df,
        x="Outcome",
        y="Event",
        color="Member",
        size=np.where(df["Member"], 18, 8),
        color_discrete_sequence=px.colors.sequential.Plasma[2:8:4],
        hover_data=["Outcome", "Event", "Member"],
    )
    return style_plotly(
        fig,
        f"{case_name} — A={sorted(event_a)}, B={sorted(event_b)}",
    )


widgets.interact(
    lambda case_name: relationship_figure(case_name).show(),
    case_name=widgets.Dropdown(
        options=list(RELATIONSHIP_CASES),
        description="Case:",
    ),
)

---
        # 13. Event Operations Simulator

        ### Learning Objective
        Define custom events and compute event algebra automatically.

        ### Mathematical Intuition

Given $A,B\subseteq S$:

$$
A\cup B,\quad
A\cap B,\quad
A-B,\quad
A^c
$$


        ### Real-world Motivation
        Custom event definitions mirror rule engines and decision logic.

        ### AI Connection
        Threshold-based AI systems combine predicates using logical set operations.

        ### Key Observations
        - Every computed result remains a subset of $S$.

In [ ]:
SIM_SPACE = set(range(1, 11))

event_a_select = widgets.SelectMultiple(
    options=sorted(SIM_SPACE),
    value=(1, 2, 3, 4),
    description="Event A:",
    rows=8,
)
event_b_select = widgets.SelectMultiple(
    options=sorted(SIM_SPACE),
    value=(4, 5, 6, 7),
    description="Event B:",
    rows=8,
)
operation_select = widgets.Dropdown(
    options=[
        "Union A ∪ B",
        "Intersection A ∩ B",
        "Difference A − B",
        "Difference B − A",
        "Complement Aᶜ",
        "Complement Bᶜ",
        "Symmetric Difference",
    ],
    description="Operation:",
)
simulator_output = widgets.Output()


def render_event_simulator(*_):
    with simulator_output:
        clear_output(wait=True)

        event_a = set(event_a_select.value)
        event_b = set(event_b_select.value)
        result = operation_result(
            event_a,
            event_b,
            SIM_SPACE,
            operation_select.value,
        )

        display(Markdown(
            "### Computed Result\n\n"
            f"- $A={sorted(event_a)}$\n"
            f"- $B={sorted(event_b)}$\n"
            f"- **{operation_select.value}** $={sorted(result)}$"
        ))

        ordered = sorted(SIM_SPACE)
        df = pd.DataFrame({
            "Outcome": ordered,
            "In A": [item in event_a for item in ordered],
            "In B": [item in event_b for item in ordered],
            "In Result": [item in result for item in ordered],
        })

        long_df = df.melt(
            id_vars="Outcome",
            var_name="Set",
            value_name="Membership",
        )

        fig = px.scatter(
            long_df,
            x="Outcome",
            y="Set",
            color="Membership",
            size=np.where(long_df["Membership"], 18, 7),
            color_discrete_sequence=px.colors.sequential.Viridis[2:8:4],
            hover_data=["Membership"],
        )
        style_plotly(fig, "Custom Event Operations").show()


for control in (event_a_select, event_b_select, operation_select):
    control.observe(render_event_simulator, names="value")

render_event_simulator()

display(
    widgets.HBox([event_a_select, event_b_select, operation_select]),
    simulator_output,
)

---
        # 14. Monte Carlo Event Simulation

        ### Learning Objective
        Estimate event probabilities experimentally and observe convergence.

        ### Mathematical Intuition

For event $E$:

$$
\hat P_n(E)
=
\frac{1}{n}
\sum_{i=1}^{n}
\mathbf{1}\{\omega_i\in E\}
$$


        ### Real-world Motivation
        Simulation estimates probabilities when exact counting or integration is difficult.

        ### AI Connection
        Monte Carlo methods appear in uncertainty estimation, reinforcement learning, Bayesian computation, and stochastic optimization.

        ### Key Observations
        - Small samples fluctuate.
- Larger samples usually stabilize near the theoretical value.

In [ ]:
def monte_carlo_figure(
    experiment: str,
    trials: int,
) -> go.Figure:
    if experiment == "Coin: at least one Head in 2 tosses":
        samples = RNG.choice(["HH", "HT", "TH", "TT"], size=trials)
        indicators = np.array(
            [sample != "TT" for sample in samples],
            dtype=float,
        )
        theoretical = 0.75
    else:
        samples = RNG.integers(1, 7, size=trials)
        indicators = (samples % 2 == 0).astype(float)
        theoretical = 0.5

    cumulative = np.cumsum(indicators) / np.arange(1, trials + 1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=np.arange(1, trials + 1),
        y=cumulative,
        mode="lines",
        name="Empirical probability",
        line=dict(width=2),
    ))
    fig.add_hline(
        y=theoretical,
        line_dash="dash",
        annotation_text=f"Theoretical={theoretical:.3f}",
    )
    fig.update_xaxes(title="Number of trials")
    fig.update_yaxes(title="Estimated P(E)", range=[0, 1])
    return style_plotly(
        fig,
        f"Monte Carlo Convergence — {experiment}",
    )


widgets.interact(
    lambda experiment, trials: monte_carlo_figure(experiment, trials).show(),
    experiment=widgets.Dropdown(
        options=[
            "Coin: at least one Head in 2 tosses",
            "Die: even outcome",
        ],
        description="Experiment:",
    ),
    trials=widgets.IntSlider(
        value=1000,
        min=100,
        max=10000,
        step=100,
        description="Trials:",
        continuous_update=False,
    ),
)

---
        # 15. Machine Learning Classification

        ### Learning Objective
        Interpret classifier outputs and confusion-matrix categories as events.

        ### Mathematical Intuition

Binary classification:

$$
S=\{0,1\}
$$

Threshold event:

$$
E_\tau
=
\{x:P(\hat y=1\mid x)\ge\tau\}
$$


        ### Real-world Motivation
        Events represent spam flags, disease predictions, approvals, and alerts.

        ### AI Connection

A true positive is an event intersection:

$$
\{\hat Y=1\}\cap\{Y=1\}
$$


        ### Key Observations
        - Classification metrics are built from event counts.
- Threshold changes redefine event membership.

In [ ]:
N_POINTS = 400
TRUE_LABELS = RNG.integers(0, 2, size=N_POINTS)
SCORES = np.clip(
    RNG.normal(
        loc=0.25 + 0.5 * TRUE_LABELS,
        scale=0.18,
        size=N_POINTS,
    ),
    0,
    1,
)


def ml_event_figure(threshold: float = 0.5) -> go.Figure:
    predicted = SCORES >= threshold

    status = np.select(
        [
            (predicted == 1) & (TRUE_LABELS == 1),
            (predicted == 1) & (TRUE_LABELS == 0),
            (predicted == 0) & (TRUE_LABELS == 1),
        ],
        ["True Positive", "False Positive", "False Negative"],
        default="True Negative",
    )

    df = pd.DataFrame({
        "Index": np.arange(N_POINTS),
        "Score": SCORES,
        "True Label": TRUE_LABELS.astype(str),
        "Outcome Event": status,
    })

    fig = px.scatter(
        df,
        x="Index",
        y="Score",
        color="Outcome Event",
        hover_data=["True Label", "Outcome Event"],
        color_discrete_sequence=px.colors.qualitative.Safe,
    )
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        annotation_text=f"Threshold τ={threshold:.2f}",
    )
    fig.update_yaxes(range=[0, 1])
    return style_plotly(
        fig,
        "Binary Classification as Event Membership",
    )


widgets.interact(
    lambda threshold: ml_event_figure(threshold).show(),
    threshold=widgets.FloatSlider(
        value=0.5,
        min=0.1,
        max=0.9,
        step=0.05,
        description="Threshold:",
        continuous_update=False,
    ),
)

---
        # 16. Bayesian Event Preview

        ### Learning Objective
        Build intuition for conditional events without deriving Bayes' Theorem.

        ### Mathematical Intuition

Conditioning on $B$ restricts attention to the part of the sample space where $B$ occurred:

$$
S\longrightarrow B
$$


        ### Real-world Motivation
        A medical question changes from “probability of disease” to “probability of disease among patients with a positive test.”

        ### AI Connection
        Evidence filters the relevant state space in Bayesian networks and probabilistic models.

        ### Key Observations
        - Conditioning filters.
- The reference population changes.
- The next chapter formalizes this.

In [ ]:
population = pd.DataFrame({
    "Person": np.arange(1, 501),
    "Disease": RNG.random(500) < 0.12,
})
population["Positive Test"] = np.where(
    population["Disease"],
    RNG.random(500) < 0.85,
    RNG.random(500) < 0.08,
)


def conditional_filter_figure(view: str) -> go.Figure:
    if view == "Full sample space S":
        filtered = population.copy()
        title = "Before conditioning: full sample space S"
    else:
        filtered = population[population["Positive Test"]].copy()
        title = "After filtering on B = {Positive Test}"

    counts = (
        filtered["Disease"]
        .map({True: "Disease A", False: "No Disease Aᶜ"})
        .value_counts()
        .rename_axis("Region")
        .reset_index(name="Count")
    )

    fig = px.bar(
        counts,
        x="Region",
        y="Count",
        color="Region",
        color_discrete_sequence=px.colors.sequential.Plasma[1:8:5],
        text="Count",
    )
    fig.update_traces(textposition="outside")
    return style_plotly(fig, title)


widgets.interact(
    lambda view: conditional_filter_figure(view).show(),
    view=widgets.ToggleButtons(
        options=["Full sample space S", "Condition on Positive Test"],
        description="View:",
    ),
)

---
        # 17. NLP Event Visualization

        ### Learning Objective
        Interpret next-token prediction as probability mass over token events.

        ### Mathematical Intuition

At one language-model step:

$$
S=\{t_1,t_2,\dots,t_V\}
$$

For a token event $E$:

$$
P(E)=\sum_{t\in E}P(t)
$$


        ### Real-world Motivation
        Language models distribute probability across many candidate continuations.

        ### AI Connection
        A semantic event can group candidate tokens by sentiment, grammar, or meaning.

        ### Key Observations
        - Token sample spaces can be huge.
- Events can group tokens semantically rather than by exact identity.

In [ ]:
TOKENS = ["great", "good", "useful", "bad", "slow", "the", "a", "model"]
TOKEN_PROBS = np.array([0.22, 0.18, 0.11, 0.07, 0.06, 0.14, 0.10, 0.12])
TOKEN_PROBS = TOKEN_PROBS / TOKEN_PROBS.sum()

TOKEN_EVENTS = {
    "Positive sentiment": {"great", "good", "useful"},
    "Negative sentiment": {"bad", "slow"},
    "Function words": {"the", "a"},
    "Content token": {"model"},
}


def nlp_event_figure(event_name: str) -> go.Figure:
    df = pd.DataFrame({
        "Token": TOKENS,
        "Probability": TOKEN_PROBS,
        "In Event": [token in TOKEN_EVENTS[event_name] for token in TOKENS],
    })

    event_mass = df.loc[df["In Event"], "Probability"].sum()

    fig = px.bar(
        df,
        x="Token",
        y="Probability",
        color="In Event",
        color_discrete_sequence=px.colors.sequential.Viridis[2:8:4],
        hover_data=["In Event"],
        text=df["Probability"].map(lambda value: f"{value:.3f}"),
    )
    fig.update_traces(textposition="outside")
    return style_plotly(
        fig,
        f"LLM Next-Token Event: {event_name} — P(E)={event_mass:.3f}",
    )


widgets.interact(
    lambda event_name: nlp_event_figure(event_name).show(),
    event_name=widgets.Dropdown(
        options=list(TOKEN_EVENTS),
        description="Event:",
    ),
)

---
        # 18. Computer Vision Event Visualization

        ### Learning Objective
        Model class detections and confidence thresholds as events.

        ### Mathematical Intuition

Suppose:

$$
S=\{\text{cat},\text{dog},\text{car},\text{person}\}
$$

A thresholded event can be:

$$
E=\{\text{person detected with confidence}\ge\tau\}
$$


        ### Real-world Motivation
        Vision systems trigger actions when selected classes are detected.

        ### AI Connection
        Autonomous vehicles and robotics define safety events from class identity and confidence.

        ### Key Observations
        - Detection events combine class membership with score thresholds.

In [ ]:
CV_OBJECTS = ["Cat", "Dog", "Car", "Person"]
CV_CONFIDENCE = np.array([0.81, 0.64, 0.92, 0.73])


def cv_event_figure(
    target: str,
    threshold: float,
) -> go.Figure:
    df = pd.DataFrame({
        "Class": CV_OBJECTS,
        "Confidence": CV_CONFIDENCE,
    })
    df["In Event"] = (
        df["Class"].eq(target)
        & (df["Confidence"] >= threshold)
    )

    fig = px.bar(
        df,
        x="Class",
        y="Confidence",
        color="In Event",
        color_discrete_sequence=px.colors.sequential.Inferno[2:8:4],
        text=df["Confidence"].map(lambda value: f"{value:.2f}"),
        hover_data=["In Event"],
    )
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        annotation_text=f"τ={threshold:.2f}",
    )
    fig.update_yaxes(range=[0, 1])
    return style_plotly(
        fig,
        f"CV Event: {{Class={target}, Confidence≥{threshold:.2f}}}",
    )


widgets.interact(
    lambda target, threshold: cv_event_figure(target, threshold).show(),
    target=widgets.Dropdown(
        options=CV_OBJECTS,
        description="Target:",
    ),
    threshold=widgets.FloatSlider(
        value=0.7,
        min=0.0,
        max=1.0,
        step=0.05,
        description="Threshold:",
        continuous_update=False,
    ),
)

---
        # 19. Interactive AI Dashboard

        ### Learning Objective
        Combine sample spaces, Venn reasoning, trees, probability, dice, coins, and cards in one control surface.

        ### Mathematical Intuition

The dashboard links:

$$
S,\quad E,\quad A\cup B,\quad A\cap B,\quad P(E)
$$


        ### Real-world Motivation
        Real analytic systems combine multiple views of the same uncertain process.

        ### AI Connection
        AI monitoring dashboards combine state spaces, event thresholds, probabilities, and model outputs.

        ### Key Observations
        - The same event can be viewed numerically, geometrically, and operationally.

In [ ]:
dashboard_view = widgets.ToggleButtons(
    options=[
        "Sample Space",
        "Venn Diagram",
        "Tree Diagram",
        "Probability Calculator",
        "Event Operations",
        "Dice Explorer",
        "Coin Explorer",
        "Playing Cards",
    ],
    description="View:",
)

dashboard_speed = widgets.IntSlider(
    value=700,
    min=200,
    max=1500,
    step=100,
    description="Speed ms:",
    continuous_update=False,
)

dashboard_reset = widgets.Button(
    description="Reset",
    button_style="warning",
    icon="refresh",
)

dashboard_output = widgets.Output()


def render_main_dashboard(*_):
    with dashboard_output:
        clear_output(wait=True)
        view = dashboard_view.value

        if view == "Sample Space":
            sample = set(DIE_SPACE)
            event = DIE_EVENTS["Prime Numbers"]
            display(Markdown(
                "### Sample Space and Event\n\n"
                f"- $S={sorted(sample)}$\n"
                f"- $E_{{prime}}={sorted(event)}$"
            ))
            dice_event_figure("Prime Numbers").show()

        elif view == "Venn Diagram":
            plot_venn_operation("Union A ∪ B")

        elif view == "Tree Diagram":
            plot_tree("Coins", "First branch")

        elif view == "Probability Calculator":
            event = DIE_EVENTS["Prime Numbers"]
            probability = len(event) / len(DIE_SPACE)
            display(Markdown(
                "$$\n"
                f"P(E)=\\frac{{{len(event)}}}{{{len(DIE_SPACE)}}}"
                f"={probability:.3f}\n"
                "$$"
            ))

        elif view == "Event Operations":
            event_a = {1, 2, 3, 4}
            event_b = {3, 4, 5, 6}
            result = event_a | event_b
            display(Markdown(
                f"$A={sorted(event_a)}$, "
                f"$B={sorted(event_b)}$, "
                f"$A\\cup B={sorted(result)}$"
            ))

        elif view == "Dice Explorer":
            dice_event_figure("Numbers Greater than 4").show()

        elif view == "Coin Explorer":
            plot_coin_event(3, "Exactly one Head").show()

        else:
            card_event_figure("Face Cards").show()


def reset_main_dashboard(_):
    dashboard_view.value = "Sample Space"
    dashboard_speed.value = 700


dashboard_view.observe(render_main_dashboard, names="value")
dashboard_speed.observe(render_main_dashboard, names="value")
dashboard_reset.on_click(reset_main_dashboard)

render_main_dashboard()

display(
    widgets.VBox([
        widgets.HBox([dashboard_view, dashboard_reset]),
        dashboard_speed,
        dashboard_output,
    ])
)

---
# Events Across Modern AI Systems

| Domain | Sample Space | Example Event |
|---|---|---|
| Machine Learning | Class labels | Predicted positive |
| Deep Learning | Network outputs | Confidence above threshold |
| Bayesian Networks | Variable assignments | Evidence configuration |
| Reinforcement Learning | State-action trajectories | Goal reached |
| Computer Vision | Detected classes | Person detected |
| Robotics | Sensor states | Obstacle nearby |
| NLP | Candidate tokens | Positive-sentiment token |
| Fraud Detection | Transaction outcomes | High-risk transaction |
| Recommendation Systems | Candidate items | Item clicked |
| LLMs | Vocabulary tokens | Token belongs to semantic subset |

For discrete spaces:

$$
P(E)=\sum_{\omega\in E}P(\omega)
$$

For continuous models with density $f$:

$$
P(E)=\int_E f(x)\,dx
$$

> The event is the bridge between a probability distribution and a question we care about.

---
        # 20. Final Summary Animation

        ### Learning Objective
        Reconstruct the chapter as one animated conceptual pipeline.

        ### Mathematical Intuition

$$
\text{Experiment}
\rightarrow
S
\rightarrow
\omega
\rightarrow
E
\rightarrow
\text{Event Algebra}
\rightarrow
P(E)
$$


        ### Real-world Motivation
        Almost every probabilistic question begins by identifying the event of interest.

        ### AI Connection
        Modern AI adds huge sample spaces, learned distributions, and semantic events.

        ### Key Observations
        - Events express questions.
- Event algebra provides logic.
- Probability quantifies uncertainty.

In [ ]:
SUMMARY_STAGES = [
    "Random Experiment",
    "Sample Space",
    "Outcomes",
    "Events",
    "Union",
    "Intersection",
    "Complement",
    "Probability",
    "Machine Learning",
    "Artificial Intelligence",
    "Large Language Models",
]


def create_summary_animation(speed_ms: int = 700) -> HTML:
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, len(SUMMARY_STAGES) + 1)
    ax.axis("off")

    labels = []
    for index, stage in enumerate(SUMMARY_STAGES):
        y_value = len(SUMMARY_STAGES) - index
        label = ax.text(
            0.5,
            y_value,
            stage,
            ha="center",
            va="center",
            fontsize=16,
            alpha=0.12,
        )
        labels.append(label)

        if index < len(SUMMARY_STAGES) - 1:
            ax.text(
                0.5,
                y_value - 0.5,
                "↓",
                ha="center",
                va="center",
                fontsize=18,
                alpha=0.35,
            )

    title = ax.set_title(
        "Events: From Randomness to Modern AI",
        fontsize=20,
        pad=20,
    )

    def update(frame: int):
        for index, label in enumerate(labels):
            label.set_alpha(1.0 if index <= frame else 0.12)
            label.set_fontweight("bold" if index == frame else "normal")
        return [*labels, title]

    summary_animation = animation.FuncAnimation(
        fig,
        update,
        frames=len(SUMMARY_STAGES),
        interval=speed_ms,
        repeat=True,
    )
    plt.close(fig)
    return HTML(summary_animation.to_jshtml())


speed_control = widgets.IntSlider(
    value=700,
    min=200,
    max=1500,
    step=100,
    description="Speed ms:",
    continuous_update=False,
)

widgets.interact(
    lambda speed_ms: create_summary_animation(speed_ms),
    speed_ms=speed_control,
)

---
# Chapter Checklist

After completing this notebook, you should be able to:

- [x] Define an event as $E\subseteq S$
- [x] Visualize events for coins, dice, and cards
- [x] Interpret union, intersection, difference, and complement
- [x] Recognize disjoint and exhaustive events
- [x] Compute $P(E)=|E|/|S|$ under equal likelihood
- [x] Read event relationships geometrically
- [x] Estimate probabilities with Monte Carlo simulation
- [x] Interpret classification outcomes as events
- [x] Understand conditioning as filtering
- [x] Model LLM token groups as probabilistic events

---

# What's Next? — Conditional Probability

The next chapter asks:

> How should an event probability change when additional information is known?

This leads to:

$$
P(A\mid B)
$$

and changes the effective reference space from $S$ to $B$.

---

# Final Insight

> **Events are the language of probability.** Every probabilistic question—from rolling a die to predicting the next token in a Large Language Model—can be expressed as an event inside a sample space.

Mastering event algebra builds the foundation for conditional probability, Bayesian inference, stochastic processes, machine learning, and modern AI.